# 6.6 · 高斯混合模型 / Gaussian Mixture Models (GMM) & EM

> **课程定位 / Where this fits**
> K-Means(6.1)是**硬分配**(一个点只属一个簇)且只能球形簇。GMM 把每个簇建成一个**多元高斯**, 用 **EM 算法**做**软分配**(一个点以概率属于各簇), 还能拟合**椭圆形、不同大小/朝向**的簇。它是生成式聚类, 也是 EM 算法最经典的应用——EM 后面在 HMM、缺失数据、主题模型里反复出现。
> GMM models each cluster as a multivariate Gaussian and uses EM for soft assignments — fitting elliptical clusters and giving membership probabilities.

> 💡 **面试相关 / Interview-relevant**
> - "EM 算法的 E 步和 M 步在做什么" ★★★★★
> - "GMM 与 K-Means 的关系" ★★★★★（K-Means 是 GMM 的硬/等方差极限）
> - "EM 为什么单调收敛(下界/Jensen)" ★★★★
> - "软分配 vs 硬分配" ★★★★
> - "covariance_type 选项 / 用 BIC 选簇数" ★★★★

---

## 学习目标 / Learning Objectives
1. 混合模型与 EM 的 E/M 两步。
2. 从零实现 GMM(看清责任/responsibility)。
3. 软分配 + 椭圆簇(K-Means 做不到)。
4. K-Means 是 GMM 的极限特例。
5. covariance_type + BIC/AIC 选簇数。

## 目录 / TOC
1. [混合模型与 EM ⭐](#1)
2. [🌋 数据: Old Faithful + 从零 EM ⭐](#2)
3. [软分配 + 椭圆簇 ⭐](#3)
4. [GMM 与 K-Means 的关系 ⭐](#4)
5. [BIC 选簇数 + covariance_type](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 混合模型与 EM ⭐ / Mixture Model & EM

假设数据由 K 个高斯**混合**生成: 先以概率 $\pi_k$ 选一个成分, 再从 $\mathcal{N}(\boldsymbol\mu_k,\boldsymbol\Sigma_k)$ 采样。密度:
$$p(\mathbf{x}) = \sum_{k=1}^{K}\pi_k\,\mathcal{N}(\mathbf{x};\boldsymbol\mu_k,\boldsymbol\Sigma_k)$$

要估 $\{\pi_k,\boldsymbol\mu_k,\boldsymbol\Sigma_k\}$ 的 MLE, 但**不知道每点属哪个成分**(隐变量)。**EM 算法**交替:

- **E 步(期望)**: 用当前参数算每点对每个成分的**责任(responsibility)** $\gamma_{ik}=\Pr(\text{成分}k\mid\mathbf{x}_i)$ —— 即软分配:
$$\gamma_{ik} = \frac{\pi_k\,\mathcal{N}(\mathbf{x}_i;\boldsymbol\mu_k,\boldsymbol\Sigma_k)}{\sum_j \pi_j\,\mathcal{N}(\mathbf{x}_i;\boldsymbol\mu_j,\boldsymbol\Sigma_j)}$$
- **M 步(最大化)**: 用责任作权重, 重估参数(加权均值/协方差/比例):
$$\boldsymbol\mu_k=\frac{\sum_i\gamma_{ik}\mathbf{x}_i}{\sum_i\gamma_{ik}},\quad \pi_k=\frac{\sum_i\gamma_{ik}}{n}$$

**为何收敛**: EM 每步都不减小数据对数似然(通过优化其下界, Jensen 不等式), 单调上升至局部最优。


<a id="2"></a>
## 2. 数据: Old Faithful + 从零 EM ⭐ / Old Faithful Geyser

**Old Faithful**: 美国黄石公园老忠实间歇泉的经典数据——每次喷发的**持续时长**与**距下次喷发的等待时间**。著名的**双峰**结构(短喷发→短等待 / 长喷发→长等待), 是 2 成分 GMM 的教科书案例。这里内联其结构。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal
sns.set_theme(style="whitegrid")

def make_faithful(seed=0):
    rng = np.random.default_rng(seed)
    # 两个真实模式: (eruptions 短~2, waiting 短~54) 和 (长~4.3, 长~80)
    a = rng.multivariate_normal([2.0, 54], [[0.07, 0.5],[0.5, 35]], 97)
    b = rng.multivariate_normal([4.3, 80], [[0.12, 0.8],[0.8, 45]], 175)
    X = np.vstack([a, b]); rng.shuffle(X)
    return pd.DataFrame(X, columns=["eruptions","waiting"])

faith = make_faithful()
print(f"Old Faithful: {faith.shape}")
fig, ax = plt.subplots(figsize=(6.5,5))
ax.scatter(faith["eruptions"], faith["waiting"], s=18, alpha=0.6)
ax.set_xlabel("喷发时长 (min)"); ax.set_ylabel("等待时间 (min)")
ax.set_title("Old Faithful: 明显双峰(短-短 / 长-长) → 2 成分 GMM")
plt.tight_layout(); plt.show()


In [ ]:
X = faith.values
def gmm_em(X, K, n_iter=100, seed=0):
    rng = np.random.default_rng(seed); n, d = X.shape
    mu = X[rng.choice(n, K, replace=False)]
    Sigma = np.array([np.cov(X.T) for _ in range(K)])
    pi = np.ones(K) / K
    ll_old = -np.inf
    for it in range(n_iter):
        # E 步: 责任
        resp = np.array([pi[k]*multivariate_normal(mu[k], Sigma[k]).pdf(X) for k in range(K)]).T
        ll = np.log(resp.sum(1)).sum()
        resp /= resp.sum(1, keepdims=True)
        # M 步: 加权重估
        Nk = resp.sum(0)
        mu = (resp.T @ X) / Nk[:,None]
        Sigma = np.array([(resp[:,k,None]*(X-mu[k])).T @ (X-mu[k]) / Nk[k] + 1e-6*np.eye(d)
                          for k in range(K)])
        pi = Nk / n
        if abs(ll - ll_old) < 1e-4: break
        ll_old = ll
    return mu, Sigma, pi, resp, ll

mu, Sigma, pi, resp, ll = gmm_em(X, 2)
print(f"从零 EM 收敛, 对数似然 {ll:.1f}")
print(f"成分比例 π = {pi.round(3)}, 均值:\n{mu.round(1)}")
from sklearn.mixture import GaussianMixture
gm = GaussianMixture(2, n_init=5, random_state=0).fit(X)
print(f"sklearn GMM 对数似然(×n) {gm.score(X)*len(X):.1f} (与从零接近)")


<a id="3"></a>
## 3. 软分配 + 椭圆簇 ⭐ / Soft Assignment & Elliptical Clusters


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# 软分配: 颜色=属于成分1的概率(连续) / soft membership
axes[0].scatter(X[:,0], X[:,1], c=resp[:,0], cmap="coolwarm", s=22)
axes[0].set_title("GMM 软分配: 颜色=成分隶属概率(边界处~0.5)")
axes[0].set_xlabel("eruptions"); axes[0].set_ylabel("waiting")

# 画椭圆等高线 / Gaussian ellipses
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 200),
                     np.linspace(X[:,1].min()-5, X[:,1].max()+5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
dens = sum(pi[k]*multivariate_normal(mu[k], Sigma[k]).pdf(grid) for k in range(2)).reshape(xx.shape)
axes[1].scatter(X[:,0], X[:,1], s=14, alpha=0.4)
axes[1].contour(xx, yy, dens, levels=8, cmap="viridis")
axes[1].scatter(mu[:,0], mu[:,1], c="red", marker="X", s=150)
axes[1].set_title("GMM 拟合椭圆高斯(有朝向/不同形状, K-Means 做不到)")
axes[1].set_xlabel("eruptions"); axes[1].set_ylabel("waiting")
plt.tight_layout(); plt.show()
print("软分配: 每点给出隶属各簇的概率; 椭圆协方差 → 拟合有朝向、不同大小的簇")


<a id="4"></a>
## 4. GMM 与 K-Means 的关系 ⭐ / GMM vs K-Means

**K-Means 是 GMM 的特例/极限**:
- 协方差固定为 $\sigma^2\mathbf{I}$(球形等方差) **且** $\sigma\to0$ → 责任退化成 0/1 硬分配, M 步均值更新就是 K-Means。
- 反过来 GMM 是"软化 + 允许椭圆"的 K-Means。

实务: 球形等大簇用 K-Means(快); 椭圆/重叠/要概率用 GMM。GMM 同样依赖初始化(常用 K-Means 初始化, sklearn 默认如此)。


In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
# 椭圆+重叠的数据上对比 / elongated overlapping clusters
rng = np.random.default_rng(3)
T = np.array([[2.5, 1.2],[0, 0.5]])
c1 = rng.normal(0,1,(250,2)) @ T + [0,0]
c2 = rng.normal(0,1,(250,2)) @ T + [3,3]
Xe = np.vstack([c1, c2])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(Xe[:,0], Xe[:,1], c=KMeans(2,n_init=10,random_state=0).fit_predict(Xe), cmap="coolwarm", s=12)
axes[0].set_title("K-Means: 球形假设 → 在拉长簇上切错")
axes[1].scatter(Xe[:,0], Xe[:,1], c=GaussianMixture(2,n_init=5,random_state=0).fit_predict(Xe), cmap="coolwarm", s=12)
axes[1].set_title("GMM: 椭圆协方差 → 正确分开拉长簇")
plt.tight_layout(); plt.show()
print("拉长/有朝向的簇: K-Means 切错, GMM 正确 → GMM 是允许椭圆+软分配的 K-Means")


<a id="5"></a>
## 5. BIC 选簇数 + covariance_type / Model Selection

GMM 是概率模型, 可用 **BIC / AIC**(信息准则, 似然 − 复杂度惩罚)选簇数——比聚类的肘部更有理论依据。**covariance_type** 控制协方差形状: `full`(各簇任意椭圆) / `diag`(轴对齐椭圆) / `tied`(共享) / `spherical`(球形=接近K-Means)。


In [ ]:
Ks = range(1, 8)
bics = [GaussianMixture(k, n_init=3, random_state=0).fit(X).bic(X) for k in Ks]
aics = [GaussianMixture(k, n_init=3, random_state=0).fit(X).aic(X) for k in Ks]
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(list(Ks), bics, "o-", label="BIC"); ax.plot(list(Ks), aics, "s--", label="AIC")
best = list(Ks)[int(np.argmin(bics))]
ax.axvline(best, color="r", ls=":", label=f"BIC 最优 K={best}")
ax.set_xlabel("成分数 K"); ax.set_ylabel("信息准则(越低越好)"); ax.legend()
ax.set_title("BIC/AIC 选簇数: Old Faithful 选出 2(符合双峰)")
plt.tight_layout(); plt.show()
print(f"BIC 最优 K = {best} (数据本就 2 个模式)")
print("covariance_type: full(任意椭圆)/diag(轴对齐)/tied(共享)/spherical(球形≈KMeans)")


<a id="6"></a>
## 6. 小结 / Summary

```
GMM: 数据=K 个高斯混合 p(x)=Σπ_k N(x;μ_k,Σ_k); 软分配(隶属概率)
EM: E步算责任 γ_ik(软分配) → M步用责任加权重估 μ,Σ,π; 单调升似然到局部最优
软分配 + 椭圆协方差 → 拟合有朝向/不同大小/重叠的簇(K-Means 做不到)
K-Means = GMM 的 球形等方差 + σ→0(硬分配)极限
选簇数用 BIC/AIC; covariance_type 控椭圆形状; 同样需好初始化(默认 K-Means 初始)
```

### 💡 面试速查
1. **EM**: E步软分配(责任 γ), M步加权重估参数; 单调升对数似然
2. **GMM=软化+椭圆的 K-Means**; K-Means 是其球形+硬分配极限
3. **软分配**给隶属概率(边界点~0.5), 可表达不确定性
4. **BIC/AIC** 选簇数(有理论依据); covariance_type 选协方差形状
5. EM 收敛到**局部**最优, 依赖初始化 + 需防协方差奇异(加 ε)

### 下一节
**6.7 谱聚类**——又一个能处理非凸簇的方法。把数据建成图, 用图拉普拉斯的特征向量降维, 再在新空间里跑 K-Means。
